# LGG MRI Segmentation — EDA

Sanity-checks before training: dataset size, tumor prevalence, slice/patient distribution, and 10 random image+mask pairs.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.dataset import discover_slices, patient_level_split, verify_no_patient_leakage

In [ ]:
DATA_ROOT = Path('../data/lgg-mri-segmentation/kaggle_3m')
records = discover_slices(DATA_ROOT)
df = pd.DataFrame([r.__dict__ for r in records])
print(df.shape)
df.head()

In [ ]:
print(f"Total slices: {len(df)}")
print(f"Patients:     {df['patient_id'].nunique()}")
print(f"Tumor slices: {df['has_tumor'].sum()} ({df['has_tumor'].mean()*100:.1f}%)")
print(f"Empty slices: {(~df['has_tumor']).sum()}")

slices_per_patient = df.groupby('patient_id').size()
tumor_per_patient = df.groupby('patient_id')['has_tumor'].sum()
print(f"Slices/patient: min={slices_per_patient.min()} median={slices_per_patient.median():.0f} max={slices_per_patient.max()}")
print(f"Tumor slices/patient: min={tumor_per_patient.min()} median={tumor_per_patient.median():.0f} max={tumor_per_patient.max()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(slices_per_patient, bins=20, color='steelblue')
axes[0].set_title('Slices per patient')
axes[0].set_xlabel('# slices')
axes[1].hist(tumor_per_patient, bins=20, color='salmon')
axes[1].set_title('Tumor slices per patient')
axes[1].set_xlabel('# tumor slices')
plt.tight_layout()

## Verify patient-level split is leak-free

In [ ]:
splits = patient_level_split(records, seed=42)
verify_no_patient_leakage(splits)
for name, recs in splits.items():
    n_pat = len({r.patient_id for r in recs})
    n_tum = sum(r.has_tumor for r in recs)
    print(f'{name:5s}: {len(recs):4d} slices / {n_pat:3d} patients / {n_tum:4d} tumor')

all_ids = {sp: {r.patient_id for r in recs} for sp, recs in splits.items()}
assert all_ids['train'].isdisjoint(all_ids['val'])
assert all_ids['train'].isdisjoint(all_ids['test'])
assert all_ids['val'].isdisjoint(all_ids['test'])
print('\nNo patient overlap across splits. ✓')

## 10 random image + mask pairs

In [ ]:
rng = np.random.default_rng(0)
tumor_records = [r for r in records if r.has_tumor]
sample = rng.choice(tumor_records, size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, rec in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(rec.image_path), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(rec.mask_path, cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[msk > 0] = (0.6 * overlay[msk > 0] + 0.4 * np.array([255, 0, 0])).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f"{rec.patient_id[:18]}\nslice {rec.slice_idx}", fontsize=9)
    ax.axis('off')
plt.tight_layout()

## Mask area distribution
If most masks are tiny, BCE alone will collapse to all-zero predictions — a key reason for the Dice + BCE combo.

In [ ]:
areas = []
for rec in tumor_records:
    msk = cv2.imread(rec.mask_path, cv2.IMREAD_GRAYSCALE)
    areas.append((msk > 0).sum())
areas = np.array(areas)
img_area = 256 * 256
print(f'Tumor area as % of image: min={areas.min()/img_area*100:.2f}% median={np.median(areas)/img_area*100:.2f}% max={areas.max()/img_area*100:.2f}%')
plt.hist(areas / img_area * 100, bins=40, color='teal')
plt.xlabel('Tumor area (% of slice)')
plt.ylabel('# slices')
plt.title('Tumor footprint distribution')